- 부모 청크: 헤딩 기준으로 자른 기존 청크 (LLM 답변용)
- 자식 청크: 항목 (1)(가)① 기준으로 추가 분할 (검색용)

질문 → 자식 청크로 검색 → 부모 청크 ID 확인 → 부모 청크 내용을 LLM에 전달

In [1]:
pdf_dir          = r"D:\GIT\large_language_model-main\pdf"
output_dir       = r"D:\GIT\large_language_model-main\output_chunks"
max_workers      = 5
min_chunk_length = 20

In [2]:
import os, re, json, hashlib, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

import pdfplumber
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

**줄 단위로 제거**
- `KOSHA GUIDE` : 문서 상단 헤더
- `D-C-7-2026` : 문서 번호
- `- 1 -` : 페이지 번호
- `<그림 1>`, `그림 1.` : 그림 캡션
- `목 차` : 목차 제목
- 개정이력, 부록 관련 텍스트

**페이지 전체를 스킵**
- 부록 페이지
- 개정이력 페이지
- 제안개요, 심의개요 페이지

`is_noise()` → 줄 하나가 노이즈인지 확인
`should_skip_page()` → 페이지 전체를 건너뛸지 확인

In [3]:
noise_patterns = [
    re.compile(r'^kosha\s+guide',                      re.I),
    re.compile(r'^d\s*[–-]\s*c\s*[–-]\s*\d',          re.I),
    re.compile(r'^-\s*\d+\s*-$'),
    re.compile(r'^<(그림|표)\s*\d+'),
    re.compile(r'^(그림|표)\s*\d+[\s.]'),
    re.compile(r'^목\s*차$'),
    re.compile(r'^기술지원규정\s*(개정\s*이력|의\s*개요)'),
    re.compile(r'^(□|◦|○)\s*(개정|제정|관련규격|공표|제정자)'),
    re.compile(r'^<부록'),
]

skip_page_pattern = re.compile(
    r'(^<부록|^기술지원규정\s*개정\s*이력|^기술지원규정의\s*개요'
    r'|^제안개요|^심의개요|^ⅰ\.\s*제정이유|^ⅱ\.\s*제정)',
    re.I | re.M
)


def is_noise(line):
    s = line.strip()
    return (
        len(s) <= 2 or
        s.count('·') >= 5 or s.count('…') >= 3 or
        any(p.search(s) for p in noise_patterns)
    )


def should_skip_page(text):
    return bool(skip_page_pattern.search(text))

`heading_pattern` : 숫자로 시작하는 줄을 헤딩으로 인식하는 정규식

`parse_heading()` : 줄이 헤딩이면 (depth, 번호, 제목) 반환, 아니면 None 반환
- 점(`.`) 개수로 depth 결정: `7` → depth 1, `7.1` → depth 2, `7.1.1` → depth 3
- depth 4 이상(`7.1.1.1`)은 헤딩으로 안보는 것으로 적용

In [4]:
heading_pattern = re.compile(
    r'^(?P<num>\d{1,2}(?:\.\d{1,2}){0,2})\.?\s+(?P<title>.+)$'
)


def parse_heading(line):
    m = heading_pattern.match(line.strip())
    if not m:
        return None
    num   = m.group('num')
    title = m.group('title').strip()
    depth = num.count('.') + 1
    if len(title) < 2 or depth > 3:
        return None
    return depth, num, title

**유사도가 0.6~0.7에서 막혔던 이유**

- 기존 청크 (내용이 너무 많이 섞임)
- 해결책 → 항목 단위로 잘게 쪼개기
자식 청크 1: "(1)..." / 
자식 청크 2: "(2)..." / 
자식 청크 3: "(3)..." / 
자식 청크 4: "(4)..."

`item_pattern` : `(1)`, `(가)`, `①` 같은 항목 시작 패턴을 감지

`split_into_children()` : 부모 청크의 content를 항목 단위로 분할해서 자식 청크 리스트 반환

In [5]:
item_pattern = re.compile(
    r'^(\(\d+\)|\([가-힣]\)|[①②③④⑤⑥⑦⑧⑨⑩]|[가-힣]\.\s)'
)


def split_into_children(content):
    lines    = content.splitlines()
    children = []
    current  = []

    for line in lines:
        if item_pattern.match(line.strip()) and current:
            text = "\n".join(current).strip()
            if len(text) >= min_chunk_length:
                children.append(text)
            current = [line]
        else:
            current.append(line)

    if current:
        text = "\n".join(current).strip()
        if len(text) >= min_chunk_length:
            children.append(text)

    return children if children else [content]

파일명에서 문서 제목 & 연도 추출

In [6]:
year_pattern = re.compile(r'(20\d{2}|19\d{2})')


def parse_filename(filename):
    stem = Path(filename).stem
    year_match = year_pattern.search(stem)
    year  = int(year_match.group()) if year_match else None
    title = re.sub(r'^[A-Za-z0-9\s\-_]+(?=\s|[가-힣])', '', stem).strip()
    title = title.replace('_', ' ').strip() or stem
    return title, year

처음메는 TASK를 OPENAPI 를 사용해서 자동으로 추출하는 방식으로 시도 -> "기타" 로 구분 되는 경우가 너무 많음

NOTION 에 설정한 TASK+@ 에 키워드 목록을 적당히 넣어 사용하는 방식으로 다시 시도함

TASK 뿐만 아니라 SPACE 도 동일하게 적용

In [7]:
TASK_KEYWORDS = {
    "비계": ["비계", "강관비계", "시스템비계", "이동식비계", "작업발판"],
    "달비계": ["달비계", "작업의자", "로프강하"],
    "곤돌라": ["곤돌라"],
    "고소작업대": ["고소작업대", "시저리프트"],
    "안전대 사용": ["안전대", "안전벨트", "추락방지대"],
    "외벽도장보수공사": ["외벽도장", "외벽보수", "외벽청소"],
    "용접": ["용접", "용단", "아크", "가스절단"],
    "철근": ["철근", "배근", "철근공"],
    "거푸집": ["거푸집", "동바리"],
    "콘크리트": ["콘크리트", "타설", "양생", "레미콘"],
    "굴착": ["굴착", "굴착작업", "터파기"],
    "터널공사": ["터널", "NATM", "쉴드", "갱도"],
    "조경공사": ["조경", "식재", "수목"],
    "양중": ["양중", "인양", "타워크레인", "이동식크레인", "크레인"],
    "해체": ["해체", "철거", "해체공사"],
    "내장공사": ["내장", "천장", "경량철골", "인테리어"],
    "교량공사": ["교량", "거더", "교각", "PSC"],
    "도장": ["도장", "도료", "페인트"],
    "가설작업": ["가설계단", "가설통로", "가설울타리"],
    "발파": ["발파", "화약", "뇌관"],
    "흙막이": ["흙막이", "엄지말뚝", "토류판", "강널말뚝"],
    "말뚝항타": ["항타", "말뚝", "천공", "CIP", "현장타설"],
    "철골공사": ["철골", "강구조", "볼트", "강재"],
    "방수공사": ["방수", "시트방수", "도막방수"],
    "조적미장": ["조적", "미장", "타일", "벽돌"],
    "포장공사": ["포장", "아스팔트", "노면"],
    "건설기계": ["굴착기", "덤프트럭", "로더", "불도저", "지게차"],
    "지붕공사": ["지붕", "슬레이트", "패널", "홈통"],
    "수상해상": ["수상", "바지선", "해상", "항만", "선박"],
    "운반": ["운반", "자재운반", "소운반"],
    "전기": ["전기", "감전", "충전부"],
    "동바리": ["동바리", "지지대", "서포트"],
    "절단": ["절단", "절단작업", "절단기"],
    "하역": ["하역", "상하차", "적재하역"],
    "밀폐공간": ["밀폐공간", "환기", "산소결핍"],
    "천공": ["천공", "천공기", "천공작업"],
    "토공": ["토공", "토사", "토공사"],
    "사다리": ["사다리", "이동식사다리", "승강"],
}


SPACE_KEYWORDS = {
    "지하": ["지하", "지하층", "지하실"],
    "상부": ["상부", "상단", "상부작업"],
    "고소": ["고소", "높이", "고소작업"],
    "하부": ["하부", "하단", "하부작업"],
    "터널": ["터널", "갱내", "갱도"],
    "내부": ["내부", "구조물 내부", "설비 내부"],
    "지붕": ["지붕", "지붕면", "옥상"],
    "바지선": ["바지선", "부선", "작업바지선"],
    "천장": ["천장", "천장면", "상부 천장"],
    "밀폐공간": ["밀폐공간", "맨홀", "탱크"],
    "개구부": ["개구부", "개구부 주변", "개구부 덮개"],
    "선박": ["선박", "선내", "선체"],
    "탱크": ["탱크", "저장탱크", "탱크 내부"],
    "갑판": ["갑판", "선박 갑판", "바지선 갑판"],
    "실내": ["실내", "건물 내부", "작업장 내부"],
    "맨홀": ["맨홀", "맨홀 내부", "맨홀 작업"],
    "지하층": ["지하층", "지하 작업장", "지하 공간"],
    "옥외": ["옥외", "야외", "노천"],
    "갱내": ["갱내", "갱 안", "터널 내부"],
    "노천": ["노천", "노천작업", "옥외작업"],
}


def assign_task(source, content):
    text = source + " " + content
    for label, keywords in TASK_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return label
    return "기타"


def assign_space(content):
    for label, keywords in SPACE_KEYWORDS.items():
        if any(kw in content for kw in keywords):
            return label
    return "옥외"

PDF 텍스트 추출 
- 위에서 설정 했던 방식으로 추출 진행

In [8]:
def extract_lines(pdf_path):
    result = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if should_skip_page(text):
                continue
            for line in text.splitlines():
                clean = line.strip()
                if clean and not is_noise(clean):
                    result.append((page_num, clean))
    return result

pdfplumber을 사용하여 표를 텍스트로 추출
RAG 챗봇이 표 안의 수치/비교 내용도 답변할 수 있도록 텍스트로 추출

- 내용이 50자 미만이면 의미없는 표로 판단해서 제외
- KOSHA GUIDE 헤더 같은 노이즈 표도 제외

In [9]:
def extract_tables(pdf_path):
    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if should_skip_page(text):
                continue
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                header = [str(cell or "").strip() for cell in table[0]]
                lines  = []
                for row in table[1:]:
                    for i, cell in enumerate(row):
                        cell_text = str(cell or "").strip()
                        if cell_text and i < len(header) and header[i]:
                            lines.append(f"{header[i]}: {cell_text}")
                content = "\n".join(lines)
                if len(content) < 50:
                    continue
                if any(p.search(content) for p in noise_patterns):
                    continue
                tables_text.append({"page": page_num, "content": content})
    return tables_text

청크 생성 (부모 + 자식)

**예시**

**부모 청크**
```json
{
  "chunk_id": "abc123",
  "chunk_type": "parent",
  "depth_1": "7 비계 안전작업계획",
  "depth_2": "7.1 강관비계",
  "depth_3": "7.1.3 조립",
  "content": "(1) 비계기둥 간격은...
(2) 띠장 수직간격은...
(3) 교차가새는..."
}
```

**자식 청크** 
```json 
{ 
  "chunk_id": "def456",
  "chunk_type": "child",
  "parent_id": "abc123 (부모 청크 ID 참조)", 
  "depth_1": "7 비계 안전작업계획",
  "content": "(1) 비계기둥 간격은 1.85m 이하..."
}
```

자식 청크로 검색하고 나서 `parent_id`로 부모 청크를 찾아서 LLM에게 전달

자식 → 검색 정확도 ↑ (짧고 명확해서 유사도 높음)
부모 → 답변 품질 ↑  (맥락이 있어서 LLM이 완전한 답변 가능)


In [10]:
def make_chunk_id(source, index):
    return hashlib.md5(f"{Path(source).stem}_{index}".encode()).hexdigest()[:12]


def build_chunks(lines, tables, pdf_path):
    filename = Path(pdf_path).name
    title, year = parse_filename(filename)

    parent_chunks = []   # (LLM 답변용)
    child_chunks  = []   # (검색용)

    current_headings = {1: ("", ""), 2: ("", ""), 3: ("", "")}
    body_lines       = []
    chunk_index      = 0

    def save_chunk():
        nonlocal chunk_index
        content = "\n".join(body_lines).strip()
        if len(content) < min_chunk_length:
            return

        d1 = f"{current_headings[1][0]} {current_headings[1][1]}".strip()
        d2 = f"{current_headings[2][0]} {current_headings[2][1]}".strip()
        d3 = f"{current_headings[3][0]} {current_headings[3][1]}".strip()

        chunk_index += 1
        parent_id = make_chunk_id(pdf_path, chunk_index)

        parent_chunks.append({
            "chunk_id"   : parent_id,
            "chunk_type" : "parent",   # LLM 답변용
            "source"     : filename,
            "title"      : title,
            "year"       : year,
            "status"     : "Active",
            "depth_1"    : d1,
            "depth_2"    : d2,
            "depth_3"    : d3,
            "category"   : "",
            "task"       : "",
            "space"      : "",
            "keyword"    : [],
            "content"    : content,
        })

        children = split_into_children(content)
        for child_content in children:
            chunk_index += 1
            child_chunks.append({
                "chunk_id"   : make_chunk_id(pdf_path, chunk_index),
                "chunk_type" : "child",    # 검색용
                "parent_id"  : parent_id,  # 부모 청크 참조
                "source"     : filename,
                "title"      : title,
                "year"       : year,
                "status"     : "Active",
                "depth_1"    : d1,
                "depth_2"    : d2,
                "depth_3"    : d3,
                "category"   : "",
                "task"       : "",
                "space"      : "",
                "keyword"    : [],
                "content"    : child_content,
            })

    for _, line in lines:
        heading = parse_heading(line)
        if heading:
            save_chunk()
            body_lines = []
            depth, num, t = heading
            current_headings[depth] = (num, t)
            for d in range(depth + 1, 4):
                current_headings[d] = ("", "")
        else:
            body_lines.append(line)

    save_chunk()

    # 표는 부모 청크로만 저장 (분할 불필요)
    for table in tables:
        chunk_index += 1
        parent_chunks.append({
            "chunk_id"   : make_chunk_id(pdf_path, chunk_index),
            "chunk_type" : "parent",
            "source"     : filename,
            "title"      : title,
            "year"       : year,
            "status"     : "Active",
            "depth_1"    : "[표]",
            "depth_2"    : "",
            "depth_3"    : "",
            "category"   : "",
            "task"       : "",
            "space"      : "",
            "keyword"    : [],
            "content"    : table["content"],
        })

    return parent_chunks, child_chunks

PDF 처리

77개의 PDF 처리를 하나씩 하는 것 보다 하나의 ipynb 파일로 처리

- 파일명 기준으로 별도 저장
   - `파일명_parent_chunks.json` : 부모 청크
   - `파일명_child_chunks.json` : 자식 청크

In [11]:
def process_one_pdf(pdf_path):
    path    = Path(pdf_path)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n▶ {path.name}")
    try:
        lines  = extract_lines(str(path))
        tables = extract_tables(str(path))
        parents, children = build_chunks(lines, tables, str(path))
        print(f"  부모: {len(parents)}개 / 자식: {len(children)}개")

        # 부모·자식 별도 저장
        for suffix, chunks in [("parent", parents), ("child", children)]:
            save_path = out_dir / f"{path.stem}_{suffix}_chunks.json"
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(
                    {"source": path.name, "total_chunks": len(chunks), "chunks": chunks},
                    f, ensure_ascii=False, indent=2
                )

        print(f"저장 완료")
        return {"status": "ok", "file": path.name,
                "parents": len(parents), "children": len(children)}
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {"status": "error", "file": path.name, "reason": str(e)}

In [12]:
pdf_files = sorted(Path(pdf_dir).glob("*.pdf"))

if not pdf_files:
    print(f" PDF 없음: {pdf_dir}")
else:
    print(f"총 {len(pdf_files)}개 PDF 추출 시작")
    results = [process_one_pdf(str(p)) for p in pdf_files]

    success = [r for r in results if r["status"] == "ok"]
    failed  = [r for r in results if r["status"] == "error"]

    print(f"\n{'='*55}")
    for r in success:
        print(f" O {r['file']} → 부모 {r['parents']}개 / 자식 {r['children']}개")
    for r in failed:
        print(f" X {r['file']} → {r['reason']}")
    print(f"\n 성공: {len(success)}/{len(results)}개")
    total_p = sum(r.get('parents',  0) for r in success)
    total_c = sum(r.get('children', 0) for r in success)
    print(f"  부모 총 {total_p}개 / 자식 총 {total_c}개")

총 77개 PDF 추출 시작

▶ 352869_C - 73 - 2012.pdf
  부모: 16개 / 자식: 123개
저장 완료

▶ C-05-2016 건설공사 돌관작업 안전보건작업 지침.pdf
  부모: 21개 / 자식: 138개
저장 완료

▶ C-06-2015 타일(Tile)공사 안전보건작업 지침.pdf
  부모: 13개 / 자식: 126개
저장 완료

▶ C-07-2012 시트(Sheet)방수 안전보건작업 지침.pdf
  부모: 9개 / 자식: 37개
저장 완료

▶ C-103-2014 굴착공사 계측관리 기술지침.pdf
  부모: 32개 / 자식: 127개
저장 완료

▶ C-108-2017 건설현장 용접용단 안전보건작업 기술지침.pdf
  부모: 18개 / 자식: 69개
저장 완료

▶ C-11-2012 가설계단  설치 및 사용 안전보건작업 지침.pdf
  부모: 18개 / 자식: 75개
저장 완료

▶ C-113-2020 취약시기 건설현장 안전작업지침.pdf
  부모: 24개 / 자식: 170개
저장 완료

▶ C-114-2020 덤프트럭 및 화물자동차 안전작업 지침.pdf
  부모: 17개 / 자식: 130개
저장 완료

▶ C-14-2012 밀폐공간의 방수공사 안전보건작업 지침.pdf
  부모: 8개 / 자식: 71개
저장 완료

▶ C-16-2016 미장공사 안전보건작업 지침.pdf
  부모: 14개 / 자식: 69개
저장 완료

▶ C-17-2011 경량철골 천장공사 안전보건작업 지침.pdf
  부모: 12개 / 자식: 73개
저장 완료

▶ C-18-2015 건설공사 안전보건 설계지침.pdf
  부모: 12개 / 자식: 104개
저장 완료

▶ C-2-2020 수상 바지(Barge)선 이용 건설공사 안전작업 지침.pdf
  부모: 13개 / 자식: 100개
저장 완료

▶ C-21-2011 현수교 교량공사 안전보건 작업지침.pdf
  부모: 15개 / 자식: 135개
저장 완료

▶ C-22-2011 사장교 교량공사 안전보건 작업지침.pdf


In [13]:
# 부모 청크 샘플로 카테고리 목록 확정
all_parents = []
for json_file in sorted(Path(output_dir).glob("*_parent_chunks.json")):
    data = json.loads(json_file.read_text(encoding="utf-8"))
    all_parents.extend(data["chunks"])
print(f"전체 부모 청크: {len(all_parents)}개")

random.seed(42)
samples = random.sample(all_parents, min(200, len(all_parents)))
sample_text = "\n---\n".join([
    f"[{c['depth_1']} > {c['depth_2']}]\n{c['content'][:200]}"
    for c in samples
])

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": """당신은 건설 안전 문서 분류 전문가입니다.
아래 건설 안전 문서 샘플들을 보고 전체 문서를 분류할 카테고리 목록을 30~40개 만들어주세요.

규칙:
- 카테고리명은 한국어 명사구 (최대 10자)
- 너무 넓지도 좁지도 않게
- '행위' 또는 '대상'이 드러나야 함
- 카테고리명만 줄바꿈으로 구분해서 출력
- 다른 말 일절 금지"""},
        {"role": "user", "content": f"건설 안전 문서 샘플:\n\n{sample_text}"}
    ],
    max_tokens=500,
    temperature=0,
)

category_list = [
    line.strip()
    for line in response.choices[0].message.content.strip().split("\n")
    if line.strip()
]

print(f"\n생성된 카테고리 목록 ({len(category_list)}개):")
for cat in category_list:
    print(f"  - {cat}")

전체 부모 청크: 2109개

생성된 카테고리 목록 (40개):
  - - 점검 행위
  - - 안전 조치
  - - 작업 계획
  - - 위험 평가
  - - 구조물 점검
  - - 장비 점검
  - - 안전 교육
  - - 작업 발판
  - - 고소 작업
  - - 굴착 작업
  - - 해체 작업
  - - 배관 공사
  - - 콘크리트 타설
  - - 철골 작업
  - - 안전 인증
  - - 화재 예방
  - - 안전 장치
  - - 개인 보호구
  - - 비계 설치
  - - 작업 환경
  - - 안전 관리
  - - 기계 운전
  - - 유해 물질 관리
  - - 전기 안전
  - - 계측 관리
  - - 작업 지휘
  - - 안전 난간 설치
  - - 낙하물 방지
  - - 구조물 해체
  - - 안전 점검
  - - 작업 구역 설정
  - - 안전 시설물 설치
  - - 작업자 안전
  - - 위험 요소 관리
  - - 안전 기준 준수
  - - 작업 절차
  - - 안전 통제
  - - 비상 대처
  - - 작업자 교육
  - - 안전 점검 기준


In [14]:
print(f"확정된 카테고리 목록 ({len(category_list)}개):")
for cat in category_list:
    print(f"  - {cat}")

확정된 카테고리 목록 (40개):
  - - 점검 행위
  - - 안전 조치
  - - 작업 계획
  - - 위험 평가
  - - 구조물 점검
  - - 장비 점검
  - - 안전 교육
  - - 작업 발판
  - - 고소 작업
  - - 굴착 작업
  - - 해체 작업
  - - 배관 공사
  - - 콘크리트 타설
  - - 철골 작업
  - - 안전 인증
  - - 화재 예방
  - - 안전 장치
  - - 개인 보호구
  - - 비계 설치
  - - 작업 환경
  - - 안전 관리
  - - 기계 운전
  - - 유해 물질 관리
  - - 전기 안전
  - - 계측 관리
  - - 작업 지휘
  - - 안전 난간 설치
  - - 낙하물 방지
  - - 구조물 해체
  - - 안전 점검
  - - 작업 구역 설정
  - - 안전 시설물 설치
  - - 작업자 안전
  - - 위험 요소 관리
  - - 안전 기준 준수
  - - 작업 절차
  - - 안전 통제
  - - 비상 대처
  - - 작업자 교육
  - - 안전 점검 기준


In [15]:
# 부모 청크에만 태깅 → 자식 청크는 부모 태그를 그대로 복사

category_str = "\n".join(category_list)

tag_system_prompt = f"""

당신은 건설 안전 문서 분류 전문가입니다.

아래 텍스트를 읽고 다음 JSON 형식으로만 답하세요. 다른 말 일절 금지.

{{"category": "...", "keyword": ["...", "...", "..."]}}

분류 기준:
- category: 아래 목록 중 하나만 선택 (목록 외 값 생성 금지)
{category_str}

- keyword: 본문에서 핵심 키워드 3~5개 추출 (명사만, 리스트 형식)

"""


def tag_chunk(depth_1, depth_2, content, source):
    task  = assign_task(source, content)
    space = assign_space(content)

    user_message = f"[{depth_1} > {depth_2}]\n{content[:500]}"
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": tag_system_prompt},
                    {"role": "user",   "content": user_message},
                ],
                max_tokens=200,
                temperature=0,
            )
            text = response.choices[0].message.content.strip()
            json_match = re.search(r'\{.*\}', text, re.DOTALL)
            result = json.loads(json_match.group() if json_match else text)
            return (
                result.get("category", "미분류"),
                task, space,
                result.get("keyword", []),
            )
        except Exception:
            if attempt < 2:
                time.sleep(3)
            else:
                return "미분류", task, space, []


def tag_all_chunks(chunks):
    total = len(chunks)
    id_to_tags = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_id = {
            executor.submit(
                tag_chunk, c["depth_1"], c["depth_2"], c["content"], c["source"]
            ): c["chunk_id"]
            for c in chunks
        }
        done = 0
        for future in as_completed(future_to_id):
            id_to_tags[future_to_id[future]] = future.result()
            done += 1
            print(f"  {done}/{total} 완료", end="\r")

    for chunk in chunks:
        cat, task, space, keyword = id_to_tags.get(
            chunk["chunk_id"], ("미분류", "기타", "옥외", [])
        )
        chunk["category"] = cat
        chunk["task"]     = task
        chunk["space"]    = space
        chunk["keyword"]  = keyword

    print(f"\n  태깅 완료")
    return chunks

In [16]:
# 부모 청크 태깅
parent_files = sorted(Path(output_dir).glob("*_parent_chunks.json"))
print(f"총 {len(parent_files)}개 파일 태깅 시작")

# 부모 chunk_id -> 태그 매핑 (자식에 복사할 때 사용)
parent_tag_map = {}

for json_file in parent_files:
    print(f"\n▷ {json_file.name}")
    data   = json.loads(json_file.read_text(encoding="utf-8"))
    chunks = tag_all_chunks(data["chunks"])
    data["chunks"] = chunks
    json_file.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    for c in chunks:
        parent_tag_map[c["chunk_id"]] = {
            "category": c["category"],
            "task"    : c["task"],
            "space"   : c["space"],
            "keyword" : c["keyword"],
        }

print(f"\n부모 청크 완료")

# 자식 청크에 부모 태그 복사
child_files = sorted(Path(output_dir).glob("*_child_chunks.json"))
print(f"\n자식 청크 태그 복사 시작: {len(child_files)}개 파일")

for json_file in child_files:
    data = json.loads(json_file.read_text(encoding="utf-8"))
    for c in data["chunks"]:
        tags = parent_tag_map.get(c.get("parent_id", ""), {})
        c["category"] = tags.get("category", "미분류")
        c["task"]     = tags.get("task",     "기타")
        c["space"]    = tags.get("space",    "옥외")
        c["keyword"]  = tags.get("keyword",  [])
    json_file.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

print("자식 태그 복사 완료")

총 77개 파일 태깅 시작

▷ 352869_C - 73 - 2012_parent_chunks.json
  16/16 완료
  태깅 완료

▷ C-05-2016 건설공사 돌관작업 안전보건작업 지침_parent_chunks.json
  21/21 완료
  태깅 완료

▷ C-06-2015 타일(Tile)공사 안전보건작업 지침_parent_chunks.json
  13/13 완료
  태깅 완료

▷ C-07-2012 시트(Sheet)방수 안전보건작업 지침_parent_chunks.json
  9/9 완료
  태깅 완료

▷ C-103-2014 굴착공사 계측관리 기술지침_parent_chunks.json
  32/32 완료
  태깅 완료

▷ C-108-2017 건설현장 용접용단 안전보건작업 기술지침_parent_chunks.json
  18/18 완료
  태깅 완료

▷ C-11-2012 가설계단  설치 및 사용 안전보건작업 지침_parent_chunks.json
  18/18 완료
  태깅 완료

▷ C-113-2020 취약시기 건설현장 안전작업지침_parent_chunks.json
  24/24 완료
  태깅 완료

▷ C-114-2020 덤프트럭 및 화물자동차 안전작업 지침_parent_chunks.json
  17/17 완료
  태깅 완료

▷ C-14-2012 밀폐공간의 방수공사 안전보건작업 지침_parent_chunks.json
  8/8 완료
  태깅 완료

▷ C-16-2016 미장공사 안전보건작업 지침_parent_chunks.json
  14/14 완료
  태깅 완료

▷ C-17-2011 경량철골 천장공사 안전보건작업 지침_parent_chunks.json
  12/12 완료
  태깅 완료

▷ C-18-2015 건설공사 안전보건 설계지침_parent_chunks.json
  12/12 완료
  태깅 완료

▷ C-2-2020 수상 바지(Barge)선 이용 건설공사 안전작업 지침_parent_chunks.json
  13/13 완료
  태깅 완